In [1]:
import pandas as pd
import geohash2

In [2]:
ais_data_df=pd.read_csv("AIS_data_small.csv")
ships_small_df=pd.read_csv("ships_small.csv")
radio_signatures_df=pd.read_csv("radio_signatures_small.csv")

In [3]:
ais_data_df['geohash'] = ais_data_df.apply(
    lambda row: geohash2.encode(row['latitude'], row['longitude'], precision=8), 
    axis=1
)
ais_data_df = ais_data_df.drop(['latitude', 'longitude'], axis=1)

In [4]:
df_merged = ais_data_df.merge(
    ships_small_df[['mmsi', 'name', 'type', 'flag', 'destination']], 
    on='mmsi', 
    how='left'          # 'left', 'inner', 'right', ou 'outer'
)
df_merged = df_merged.merge(radio_signatures_df, on='mmsi', how='left')

In [20]:
df_merged.columns

Index(['mmsi', 'timestamp_x', 'speed', 'course', 'status', 'ais_active',
       'geohash', 'name', 'type', 'flag', 'destination', 'signature_id',
       'frequency', 'bandwidth', 'modulation', 'power', 'timestamp_y',
       'location_lat', 'location_lon', 'signal_strength'],
      dtype='str')

In [24]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
import joblib

# --- 1. PRÉPARATION DES DONNÉES ---
features_cols = [
    'speed', 'course', 'status', 'ais_active', 'frequency', 
    'bandwidth', 'modulation', 'power', 'location_lat', 
    'location_lon', 'signal_strength'
]
targets_cols = ['geohash', 'name', 'type', 'flag', 'destination']

# On sépare X et y
X = df_merged[features_cols].copy()
y = df_merged[targets_cols].copy()

# Remplissage des valeurs manquantes
X = X.fillna("UNKNOWN")
y = y.fillna("UNKNOWN")

# --- 2. ENCODAGE DES FEATURES (OrdinalEncoding) ---
# On identifie les colonnes textuelles dans les features
cat_features_names = X.select_dtypes(include=['object']).columns.tolist()

feature_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[cat_features_names] = feature_encoder.fit_transform(X[cat_features_names])

# --- 3. ENCODAGE DES CIBLES (LabelEncoding) ---
# Chaque colonne cible doit être transformée en nombres
target_encoders = {}
for col in targets_cols:
    le = LabelEncoder()
    y[col] = le.fit_transform(y[col].astype(str))
    target_encoders[col] = le

# --- 4. ENTRAÎNEMENT DU MODÈLE ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# On définit CatBoost SANS cat_features (car on a déjà encodé en nombres)
base_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    verbose=0,
    loss_function='MultiClass'
)

# Utilisation du MultiOutput
model = MultiOutputClassifier(base_model, n_jobs=-1)
model.fit(X_train, y_train)

print("Entraînement terminé avec succès.")

# --- 5. SAUVEGARDE COMPLÈTE ---
# Il est crucial de sauvegarder les encodeurs pour pouvoir prédire sur de nouvelles données
save_payload = {
    'model': model,
    'feature_encoder': feature_encoder,
    'target_encoders': target_encoders,
    'features_names': features_cols,
    'cat_features_names': cat_features_names
}

joblib.dump(save_payload, 'catboost_ship_model_v2.joblib')
print("Modèle et encodeurs sauvegardés dans 'catboost_ship_model_v2.joblib'")

Entraînement terminé avec succès.
Modèle et encodeurs sauvegardés dans 'catboost_ship_model_v2.joblib'


In [26]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

predictions = model.predict(X_test)
def evaluate_multioutput(y_true, y_pred, target_names):
    for i, col in enumerate(target_names):
        print(f"\n--- Évaluation de la cible : {col} ---")
        # On utilise les encodeurs du joblib précédent pour retrouver les vrais noms si besoin
        print(classification_report(y_true.iloc[:, i], y_pred[:, i]))

evaluate_multioutput(y_test, predictions, targets_cols)


--- Évaluation de la cible : geohash ---


ValueError: Found input variables with inconsistent numbers of samples: [4, 1]

In [27]:
print(f"Nombre de lignes dans le test (réel) : {y_test.shape[0]}")
print(f"Nombre de lignes dans les prédictions : {predictions.shape[0]}")

Nombre de lignes dans le test (réel) : 4
Nombre de lignes dans les prédictions : 1
